In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import mahalanobis

# Load CSV file
df = pd.read_csv(r"C:\Users\psvig\DNN Project\MD_HI\IMS_4 features for MD.csv")

# Select the 4 feature columns
features = df.iloc[:, :4].values

# Use first 500 healthy samples
healthy_data = features[:500]

mean = healthy_data.mean(axis=0)
std = healthy_data.std(axis=0, ddof=1)
std[std == 0] = 1

healthy_norm = (healthy_data - mean) / std
full_norm = (features - mean) / std

# Healthy statistics
mean_vector = healthy_norm.mean(axis=0)
cov_matrix = np.cov(healthy_norm, rowvar=False)

# Regularization
cov_matrix += np.eye(4) * 1e-4
inv_cov = np.linalg.pinv(cov_matrix)

# Calculate Mahalanobis distance
md = np.array([
    mahalanobis(sample, mean_vector, inv_cov)
    for sample in full_norm
])

# Normalize HI between 0 and 1
hi = (md - md.min()) / (md.max() - md.min() + 1e-10)

# Plot HI
time = np.arange(1, len(hi) + 1)

plt.plot(time, hi, 'bo-', markersize=4)
plt.xlabel("Time (days)")
plt.ylabel("Health Indicator")
plt.title("Mahalanobis Distance Health Indicator")
plt.grid(True)
plt.tight_layout()
plt.show()

# Save results
pd.DataFrame({"Time": time,"HealthIndicator": hi}).to_csv("Health_indicator_MD.csv", index=False)